In [ ]:
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q

!pip install torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-2.4.0+cu121.html -q

!pip install setproctitle kaggle -q

print("Done — now run Cell 2")

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import torch
print("PyTorch:", torch.__version__)  # 2.4.0+cu121
print("GPU:", torch.cuda.get_device_name(0))

import torch_sparse
test = torch_sparse.SparseTensor.from_dense(torch.ones(3,3).cuda())
print("torch_sparse: OK")

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = ''           # write ur kaggle username
os.environ['KAGGLE_KEY']      = ''           # write ur kaggle key
DATASET_ID = f"{os.environ['KAGGLE_USERNAME']}/gowalla-attention"
print("Dataset:", DATASET_ID)

In [ ]:
import os
import subprocess

os.environ['KAGGLE_USERNAME'] = ''              # write ur kaggle username
os.environ['KAGGLE_KEY']      = ''              # write ur kaggle key

DATASET_ID  = ''                                 # write ur dataset ID
RESTORE_DIR = '/kaggle/working/restored'
os.makedirs(RESTORE_DIR, exist_ok=True)

result = subprocess.run([
    'kaggle', 'datasets', 'files', '-d', DATASET_ID
], capture_output=True, text=True)

if 'Model.py' in result.stdout or 'outModels' in result.stdout:
    print("Previous checkpoint found — downloading...")
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', DATASET_ID,
        '-p', RESTORE_DIR, '--unzip'
    ], check=True)
    print("Downloaded:", os.listdir(RESTORE_DIR))
else:
    print("Fresh start — no previous checkpoint")

In [ ]:
import os
import subprocess
import shutil

os.chdir('/kaggle/working')
if not os.path.exists("LightGNN"):
    subprocess.run(['git', 'clone', 'https://github.com/HKUDS/LightGNN.git'], check=True)
os.chdir('/kaggle/working/LightGNN')

# Download pre-trained models (if not already done)
subprocess.run(['bash', './inModels/download_demo_ckpts.sh'], check=True)

# Verify Gowalla teacher models
teacher_int = './inModels/gowalla/example_intermediate_KD_model_dim64_gowalla_ckpt.mod'
teacher_full = './inModels/gowalla/example_teacher_model_dim64_gowalla_ckpt.mod'
print("Gowalla Intermediate teacher:", "OK ✅" if os.path.exists(teacher_int) else "MISSING ❌")
print("Gowalla Full teacher:", "OK ✅" if os.path.exists(teacher_full) else "MISSING ❌")

# Verify Gowalla dataset
dataset = './Datasets/sparse_gowalla'
print("\nGowalla dataset:", "OK ✅" if os.path.exists(dataset) else "MISSING ❌")
if os.path.exists(dataset):
    print("Files:", os.listdir(dataset))

In [ ]:
import os
import shutil

os.chdir('/kaggle/working/LightGNN')

# Move extracted teacher models to correct location
for dataset in ['amazon', 'gowalla', 'yelp']:
    src_dir = f'./{dataset}'
    tgt_dir = f'./inModels/{dataset}'
    
    if os.path.exists(src_dir):
        os.makedirs(tgt_dir, exist_ok=True)
        for f in os.listdir(src_dir):
            shutil.move(os.path.join(src_dir, f), os.path.join(tgt_dir, f))
        print(f"✅ Moved {dataset} teacher models to {tgt_dir}")
    else:
        print(f"❌ {src_dir} not found")

# Verify Gowalla teacher models
teacher_int = './inModels/gowalla/example_intermediate_KD_model_dim64_gowalla_ckpt.mod'
teacher_full = './inModels/gowalla/example_teacher_model_dim64_gowalla_ckpt.mod'
print("\nGowalla Intermediate teacher:", "OK ✅" if os.path.exists(teacher_int) else "MISSING ❌")
print("Gowalla Full teacher:", "OK ✅" if os.path.exists(teacher_full) else "MISSING ❌")

In [ ]:
import os
import pickle
import scipy.sparse as sp

os.chdir('/kaggle/working/LightGNN')

data_dir = './Datasets/sparse_gowalla'

print("Loading gowalla dataset...\n")

with open(os.path.join(data_dir, 'trnMat.pkl'), 'rb') as f:
    trn_mat = pickle.load(f)

print(f"Training matrix shape: {trn_mat.shape}")
print(f"Training non-zero entries (interactions): {trn_mat.nnz}")

# Check for test and validation files
tst_interactions = 0
val_interactions = 0

tst_file = os.path.join(data_dir, 'tstMat.pkl')
if os.path.exists(tst_file):
    with open(tst_file, 'rb') as f:
        tst_mat = pickle.load(f)
    tst_interactions = tst_mat.nnz
    print(f"Test matrix shape: {tst_mat.shape}")
    print(f"Test non-zero entries: {tst_interactions}")

val_file = os.path.join(data_dir, 'valMat.pkl')
if os.path.exists(val_file):
    with open(val_file, 'rb') as f:
        val_mat = pickle.load(f)
    val_interactions = val_mat.nnz
    print(f"Validation non-zero entries: {val_interactions}")

total_interactions = trn_mat.nnz + tst_interactions + val_interactions

print("\n" + "="*60)
print("DATASET STATISTICS (from your files)")
print("="*60)
print(f"Users:          {trn_mat.shape[0]}")
print(f"Items:          {trn_mat.shape[1]}")
print(f"Interactions:   {total_interactions}")
print(f"Density:        {total_interactions / (trn_mat.shape[0] * trn_mat.shape[1]):.4e}")

In [ ]:
%%bash
echo "=== inModels folder ==="
ls /kaggle/working/LightGNN/inModels/

echo "=== gowalla folder ==="
ls /kaggle/working/LightGNN/inModels/gowalla/ 2>/dev/null || echo "gowalla folder missing"

echo "=== all .mod files ==="
find /kaggle/working/LightGNN/inModels -name "*.mod"

In [ ]:
import os
import shutil

RESTORE_DIR = '/kaggle/working/restored'
WORK_DIR    = '/kaggle/working/LightGNN'

if os.path.exists(RESTORE_DIR) and len(os.listdir(RESTORE_DIR)) > 0:
    print("Contents:", os.listdir(RESTORE_DIR))

    for fname in ['Model.py', 'Params.py', 'results_log_gowalla.txt']:
        src = os.path.join(RESTORE_DIR, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(WORK_DIR, fname))
            print(f"Restored: {fname}")

    # Restore checkpoint
    ckpt_dst = os.path.join(WORK_DIR,
        'outModels/gowalla/attention_full/checkpoints')
    os.makedirs(ckpt_dst, exist_ok=True)
    os.makedirs(os.path.join(WORK_DIR,
        'outModels/gowalla/attention_full/history'), exist_ok=True)

    for root, dirs, files in os.walk(RESTORE_DIR):
        for f in files:
            if f.endswith('.mod') and 'example_intermediate' not in f \
               and 'example_teacher' not in f:
                src = os.path.join(root, f)
                dst = os.path.join(ckpt_dst, f)
                shutil.copy2(src, dst)
                size = os.path.getsize(dst)/1024/1024
                print(f"Restored checkpoint: {f} ({size:.1f} MB)")

    print("\nCheckpoints:")
    os.system(f'ls -lth {ckpt_dst}')
else:
    print("No restore folder — fresh run")
    os.makedirs(os.path.join(WORK_DIR,
        'outModels/gowalla/attention_full/checkpoints'), exist_ok=True)
    os.makedirs(os.path.join(WORK_DIR,
        'outModels/gowalla/attention_full/history'), exist_ok=True)

In [ ]:
import os
import subprocess

os.chdir('/kaggle/working/LightGNN')

with open('Params.py', 'r') as f:
    content = f.read()

if 'use_node_attention' in content:
    print("Params.py already patched — skipping")
else:
    print("Patching Params.py...")
    with open('Params.py', 'r') as f:
        lines = f.readlines()
    clean_lines = [l for l in lines if 'use_node_attention' not in l]
    last_idx = max(i for i, l in enumerate(clean_lines)
                   if 'parser.add_argument' in l)
    indent = len(clean_lines[last_idx]) - len(clean_lines[last_idx].lstrip())
    pad = ' ' * indent
    new_line = (pad +
        "parser.add_argument('--use_node_attention', type=int, "
        "default=0, help='1=attention 0=original')\n")
    clean_lines.insert(last_idx + 1, new_line)
    with open('Params.py', 'w') as f:
        f.writelines(clean_lines)
    r = subprocess.run(
        ['python', '-m', 'py_compile', 'Params.py'],
        capture_output=True, text=True)
    print("Params.py OK" if r.returncode == 0 else r.stderr)

In [ ]:
import os
import subprocess

os.chdir('/kaggle/working/LightGNN')

with open('Model.py', 'r') as f:
    content = f.read()

if 'NodeAttention' in content:
    print("Model.py already patched — skipping")
else:
    print("Patching Model.py...")

    attention_class = '''
class NodeAttention(nn.Module):
    def __init__(self, latdim):
        super(NodeAttention, self).__init__()
        self.attn_fc = nn.Linear(2 * latdim, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(negative_slope=0.2)
        nn.init.xavier_uniform_(self.attn_fc.weight)

    def forward(self, embeds, adj):
        row, col, val = adj.coo()
        dst, src = row, col
        e_cat = torch.cat([embeds[dst], embeds[src]], dim=1)
        raw_attn = self.leaky_relu(self.attn_fc(e_cat)).squeeze(-1)
        max_vals = torch.zeros(embeds.size(0),
                               dtype=raw_attn.dtype,
                               device=raw_attn.device)
        max_vals.scatter_reduce_(0, dst, raw_attn,
                                 reduce="amax", include_self=True)
        exp_scores = torch.exp(raw_attn - max_vals[dst])
        sum_exp = torch.zeros(embeds.size(0),
                              dtype=raw_attn.dtype,
                              device=raw_attn.device)
        sum_exp.scatter_add_(0, dst, exp_scores)
        attn = exp_scores / (sum_exp[dst] + 1e-9)
        out = torch.zeros_like(embeds)
        out.scatter_add_(0,
            dst.unsqueeze(1).expand_as(embeds[src]),
            attn.unsqueeze(1) * embeds[src])
        return out

'''

    content = content.replace(
        'class Model(nn.Module):',
        attention_class + 'class Model(nn.Module):')

    old_init = '''		self.adj_nonzero = adj.nnz()
		self.emb_mask2_fixed = nn.Parameter(torch.ones(args.item+args.user, args.latdim), requires_grad=False)'''

    new_init = '''		self.adj_nonzero = adj.nnz()
		self.emb_mask2_fixed = nn.Parameter(torch.ones(args.item+args.user, args.latdim), requires_grad=False)
		self.use_node_attention = bool(args.use_node_attention)
		if self.use_node_attention:
			self.attn_layers = nn.ModuleList([NodeAttention(args.latdim) for _ in range(args.gnn_layer)])
			print(f"[NodeAttention] ENABLED - {args.gnn_layer} layers, latdim={args.latdim}")
		else:
			print("[NodeAttention] Disabled - original LightGNN")'''

    content = content.replace(old_init, new_init)

    old_fwd = '''		embedsLst = [embeds]
		for gcn in self.net_layer:
			embeds = gcn(adj, embedsLst[-1])			
			embedsLst.append(embeds)'''

    new_fwd = '''		embedsLst = [embeds]
		for i, gcn in enumerate(self.net_layer):
			gcn_out = gcn(adj, embedsLst[-1])
			if getattr(self, "use_node_attention", False):
				embeds = gcn_out + self.attn_layers[i](embedsLst[-1], adj)
			else:
				embeds = gcn_out
			embedsLst.append(embeds)'''

    content = content.replace(old_fwd, new_fwd)

    with open('Model.py', 'w') as f:
        f.write(content)

    r = subprocess.run(
        ['python', '-m', 'py_compile', 'Model.py'],
        capture_output=True, text=True)
    print("Model.py OK" if r.returncode == 0 else r.stderr)

In [ ]:
import os
os.chdir('/kaggle/working/LightGNN')

for path in [
    './outModels/gowalla/attention_full/checkpoints',
    './outModels/gowalla/attention_full/history',
]:
    os.makedirs(path, exist_ok=True)
print("Folders ready")

In [ ]:
%%bash
ls -lth /kaggle/working/LightGNN/outModels/gowalla/attention_full/checkpoints/

In [ ]:
import os

# LightGNN stores datasets here, not in 'data/'
dataset_dir = './Datasets/sparse_gowalla'

if os.path.exists(dataset_dir):
    print(f"✅ gowalla dataset folder found: {dataset_dir}")
    files = os.listdir(dataset_dir)
    print("Files:", files)
    
    # Check for required .pkl files
    required = ['trnMat.pkl', 'tstMat.pkl', 'valMat.pkl']
    for f in required:
        if f in files:
            print(f"  ✅ {f} exists")
        else:
            print(f"  ❌ {f} MISSING")
else:
    print(f"❌ gowalla dataset folder NOT found: {dataset_dir}")
    print("   Check if you ran the dataset download or if the path is different")

In [ ]:
import os

# ── Update these two lines each session ──
PRUNING_START = 1
BREAK_POINT   = None


if BREAK_POINT and os.path.exists(BREAK_POINT):
    size = os.path.getsize(BREAK_POINT)/1024/1024
    print(f"Checkpoint: {os.path.basename(BREAK_POINT)} ({size:.1f} MB)")
elif BREAK_POINT:
    print(f"ERROR — file not found: {BREAK_POINT}")
else:
    print("Fresh start — no checkpoint")

print(f"\n>>> PRUNING_START = {PRUNING_START}")
print(f">>> BREAK_POINT   = {BREAK_POINT}")

In [ ]:
import os
import subprocess
from datetime import datetime

os.chdir('/kaggle/working/LightGNN')

CKPT_PATH = './outModels/gowalla/attention_full/checkpoints/fnl_student_ckpts'
LOG_FILE  = './results_log_gowalla.txt'

print(f"Starting PRN: {PRUNING_START}")
print(f"Checkpoint:   {BREAK_POINT}")

cmd = [
    'python', 'Main.py',
    '--dataset=gowalla',
    '--mask_epoch=100',
    '--batch=4096',
    '--lr=0.001',
    '--reg=1e-8',
    '--latdim=64',
    '--gnn_layer=2',
    '--adj_aug=True',
    '--adj_aug_layer=1',
    '--use_adj_mask_aug=True',
    '--use_SM_edgeW2aug=True',
    '--adj_mask_aug1=0.05',
    '--adj_mask_aug2=1.0',
    '--use_mTea2drop_edges=True',
    '--use_tea2drop_edges=False',
    f'--PRUNING_START={PRUNING_START}',
    '--PRUNING_END=13',
    f'--model_save_path={CKPT_PATH}',
    '--his_save_path=./outModels/gowalla/attention_full/history/fnl_student_his',
    '--middle_teacher_model=./inModels/gowalla/example_intermediate_KD_model_dim64_gowalla_ckpt.mod',
    '--distill_from_middle_model=True',
    '--distill_from_teacher_model=False',
    '--train_middle_model=False',
    '--pruning_percent_adj=0.15',
    '--pruning_percent_emb=0.2',
    '--use_node_attention=1',
    '--gpu=0',
]

if BREAK_POINT:
    cmd += [
        '--train_stu_from_break_point=True',
        f'--break_point={BREAK_POINT}',
    ]

log = open(LOG_FILE, 'a')
log.write(f"\n{'='*60}\n")
log.write(f"Session started: {datetime.now()}\n")
log.write(f"PRUNING_START: {PRUNING_START}\n")
log.write(f"Break point: {BREAK_POINT}\n")
log.write(f"{'='*60}\n")
log.flush()

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True, bufsize=1
)

for line in process.stdout:
    print(line, end='')
    if ('Steps ' in line and '/347' in line) or \
       ('Step ' in line and '/72' in line):
        pass
    else:
        log.write(line)
        log.flush()

process.wait()
log.write(f"Session ended: {datetime.now()}\nReturn code: {process.returncode}\n")
log.close()
print("\nDone — return code:", process.returncode)

In [ ]:
with open('/kaggle/working/LightGNN/results_log_gowalla.txt', 'r') as f:
    lines = f.readlines()

# Print the final lines (the tail of the log)
for line in lines[-30:]:
    print(line, end='')